In [ ]:
#| default_exp review

## Reviewing notebooks

Diff and style feedback that focuses on code cells.

Review should focus on the behavior encoded in code cells, not on notebook metadata churn. This notebook keeps that review loop small: run fast.ai style hints when desired, and print nbdev-style code diffs when comparing notebooks.

Review tools draw a line between source changes and notebook noise. `style_check` is useful before publishing exported code because it combines fast.ai style hints with notebook hygiene reports, while `diff_nb` is useful during agent edits because it ignores outputs and metadata unless metadata is the only thing that changed.


In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import subprocess as _subprocess
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell as _mk_cell, read_nb as _read_nb, write_nb as _write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.review import code_source

In [ ]:
print("code cell:", code_source(_mk_cell("answer = 42", cell_type="code")))
print("markdown cell:", code_source(_mk_cell("Some docs", cell_type="markdown")))

code cell: answer = 42
markdown cell: None


In [ ]:
#| export
import ast
import glob
import json
import os
import subprocess
import sys
from pathlib import Path

from chkstyle.core import main as _chkstyle_main
from fastcore.nbio import read_nb as _read_nb
from fastcore.script import Param, call_parse
from nbdev.diff import nbs_pair, source_diff

from nbskill.foundation import (
    _empty_failure_map, _failure_map_path, _load_failure_map, cell_class_names,
    cell_source, cli_error, cli_return, is_export_directive, is_exported_code_cell,
    none_if_string, tracked_call,
)


### Style feedback as a tool

`style_check` wraps the fast.ai style checker so it can be called from the CLI or MCP without each caller rebuilding command arguments or handling strict mode.

In [ ]:
#| export
def _style_check_argv(path=".", skip_folder_re=None, skip_path=None):
    path = "." if path is None else str(path)
    argv = ["style_check", path]
    if skip_folder_re: argv += ["--skip-folder-re", str(skip_folder_re)]
    if skip_path: argv += ["--skip-path", str(skip_path)]
    return argv


def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts


def _notebook_paths(path="."):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file() and pth.suffix == ".ipynb":
        candidates = [pth]
    else:
        candidates = []
    return sorted({candidate for candidate in candidates if _is_notebook_path(candidate)})


def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not is_export_directive(line))


def _parse_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return None


def _top_level_function_count(tree):
    return sum(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) for node in tree.body)


def _assert_count(tree):
    return sum(isinstance(node, ast.Assert) for node in ast.walk(tree))


def _test_function_count(tree):
    return sum(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body
    )


def _import_keys(tree):
    keys = []
    for node in tree.body:
        if isinstance(node, ast.Import):
            for alias in node.names:
                local = alias.asname or alias.name.split(".", 1)[0]
                keys.append(f"import {alias.name} as {local}")
        elif isinstance(node, ast.ImportFrom):
            module = "." * node.level + (node.module or "")
            for alias in node.names:
                local = alias.asname or alias.name
                keys.append(f"from {module} import {alias.name} as {local}")
    return keys


def _problem_line(kind, path, cell, detail):
    return f"- {kind}: {path} id={getattr(cell, 'id', '')} {detail}"


In [ ]:
#| export
def _notebook_style_problem_lines(path="."):
    lines = []
    duplicate_imports = {}
    for nb_path in _notebook_paths(path):
        nb = _read_nb(nb_path)
        imports_by_scope = {"exported": {}, "internal": {}}
        for cell in nb.cells:
            classes = cell_class_names(cell)
            if "unclean_cell" in classes:
                visible = ", ".join(name for name in classes if name != "unclean_cell")
                lines.append(_problem_line("unclean-cell", nb_path, cell, f"semantic_types={visible}"))
            tree = _parse_code_cell(cell)
            if tree is None: continue
            source_lines = _source_without_directives(cell_source(cell)).splitlines()
            line_count = len(source_lines)
            function_count = _top_level_function_count(tree)
            if function_count > 2:
                lines.append(_problem_line("large-cell", nb_path, cell, f"{function_count} top-level functions"))
            if line_count > 20:
                lines.append(_problem_line("large-cell", nb_path, cell, f"{line_count} non-directive lines"))
            if "test_cell" in classes:
                problem_count = _assert_count(tree) + _test_function_count(tree)
                if problem_count > 1:
                    lines.append(_problem_line("multi-problem-test", nb_path, cell, f"{problem_count} asserts/test functions; split into one-problem-at-a-time cells"))
            scope = "exported" if is_exported_code_cell(cell) else "internal"
            for key in _import_keys(tree):
                imports_by_scope[scope].setdefault(key, []).append(getattr(cell, "id", ""))
        for scope, imports in imports_by_scope.items():
            for key, ids in imports.items():
                if len(ids) > 1:
                    duplicate_imports.setdefault(str(nb_path), []).append((scope, key, ids))
    for nb_path, items in duplicate_imports.items():
        for scope, key, ids in items:
            lines.append(f"- duplicate-import: {nb_path} scope={scope} import={key!r} cells={', '.join(ids)}")
    if _notebook_paths(path):
        from nbskill.graph import notebook_order_problem_lines
        lines.extend(notebook_order_problem_lines(path))
    return lines


def _format_notebook_style_report(path="."):
    lines = _notebook_style_problem_lines(path)
    if not lines: return "Notebook style report: no notebook hygiene problems found."
    return "\n".join(["Notebook style report:", *lines])


In [ ]:
#| export
def _format_count_group(title, counts):
    if not counts: return [f"{title}: none"]
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [f"{title}: " + ", ".join(f"{tool}={count}" for tool, count in ordered)]


def _format_failure_event(event):
    kind = event.get("kind", "event")
    tool = event.get("tool", "unknown")
    path = event.get("path")
    detail = event.get("summary") or event.get("error") or ",".join(event.get("reasons", []))
    location = f" path={path}" if path else ""
    return f"- {kind}: {tool}{location} {detail}".rstrip()


def _format_global_usage_summary():
    path = _failure_map_path()
    if not path.exists(): return f"Global nbskill usage: no records at {path}"
    data = _load_failure_map(path)
    counts = data.get("counts", {})
    lines = [f"Global nbskill usage: {path}"]
    lines += _format_count_group("usage", counts.get("usage", {}))
    lines += _format_count_group("failures", counts.get("failures", {}))
    lines += _format_count_group("friction", counts.get("friction", {}))
    problems = [event for event in data.get("events", []) if event.get("kind") in {"failure", "friction"}][-5:]
    if problems:
        lines.append("recent problems:")
        lines.extend(_format_failure_event(event) for event in problems)
    else:
        lines.append("recent problems: none")
    return "\n".join(lines)


def _reset_global_usage_summary():
    path = _failure_map_path()
    try: path.unlink()
    except FileNotFoundError: pass
    except OSError: path.write_text(json.dumps(_empty_failure_map(), indent=2, sort_keys=True), encoding="utf-8")


In [ ]:
#| export
def run_style_check(path=".", skip_folder_re=None, skip_path=None, strict=False):
    status = _chkstyle_main(_style_check_argv(path, skip_folder_re, skip_path))
    return status


def _normalize_style_check_cli_aliases():
    aliases = {
        "--delete-after-output": "--delete_after_output",
        "--delete-after-outout": "--delete_after_outout",
    }
    sys.argv[:] = [aliases.get(arg, arg) for arg in sys.argv]


_normalize_style_check_cli_aliases()


@call_parse
@tracked_call
def style_check(
    path: Param("File or folder to check", str, opt=False, nargs="?") = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Folder name/path to skip
    strict: bool = False,  # Exit non-zero when style hints or notebook hygiene problems are found
    delete_after_output: bool = False,  # Reset ~/.nbskill-errors.json after printing the global summary
    delete_after_outout: bool = False,  # Backward-compatible typo alias for delete_after_output
):
    "Print fast.ai style hints, notebook hygiene warnings, and global tool usage."
    status = run_style_check(path, skip_folder_re, skip_path, strict=False)
    report = _format_notebook_style_report(path)
    usage = _format_global_usage_summary()
    text = f"{report}\n\n{usage}"
    print(text)
    if delete_after_output or delete_after_outout: _reset_global_usage_summary()
    has_notebook_problems = bool(_notebook_style_problem_lines(path))
    if strict and (status or has_notebook_problems): raise SystemExit(status or 1)
    return cli_return(status or int(has_notebook_problems))


def code_source(cell): return cell.source if cell.cell_type == "code" else None


### Code-cell diffs

Notebook diffs are noisy when metadata and outputs are included. `diff_nb` asks nbdev for code-cell source on each side of a comparison and prints only the added, changed, or deleted code blocks the caller requested.

In [ ]:
#| export
def _git_ref_path_error(path, ref):
    if ref is None: return None
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0:
        return f"No git repository found for {str(path)!r}."
    root = Path(root_cmd.stdout.strip())
    try:
        rel = path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError:
        return f"{str(path)!r} is outside git repository {str(root)!r}."
    spec = f"{ref}:{rel}"
    exists_cmd = subprocess.run(
        ["git", "-C", str(root), "cat-file", "-e", spec],
        capture_output=True, text=True,
    )
    if exists_cmd.returncode == 0: return None
    return (
        f"Could not find notebook {rel!r} at git ref {ref!r}. "
        "The notebook may be new relative to that ref, or the repository may not have a HEAD commit yet. "
        "Commit the notebook first, choose an existing ref/path, or pass --ref_a None to compare against the working tree."
    )


def _git_root_rel(path):
    path = Path(path)
    root_cmd = subprocess.run(
        ["git", "-C", str(path.parent), "rev-parse", "--show-toplevel"],
        capture_output=True, text=True,
    )
    if root_cmd.returncode != 0: return None, None
    root = Path(root_cmd.stdout.strip())
    try: return root, path.resolve().relative_to(root.resolve()).as_posix()
    except ValueError: return None, None


def _notebook_json_at_ref(path, ref):
    path = Path(path)
    if ref is None:
        return json.loads(path.read_text(encoding="utf-8"))
    root, rel = _git_root_rel(path)
    if root is None: return None
    show = subprocess.run(["git", "-C", str(root), "show", f"{ref}:{rel}"], capture_output=True, text=True)
    if show.returncode != 0: return None
    return json.loads(show.stdout)


def _nbskill_metadata_by_cell(nb_json):
    cells = (nb_json or {}).get("cells", [])
    return {
        cell.get("id", str(idx)): (cell.get("metadata", {}) or {}).get("nbskill")
        for idx, cell in enumerate(cells)
    }


def _nbskill_metadata_change_count(path, ref_a, ref_b):
    try:
        old = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_a))
        new = _nbskill_metadata_by_cell(_notebook_json_at_ref(path, ref_b))
    except (OSError, json.JSONDecodeError, TypeError):
        return 0
    keys = set(old) | set(new)
    return sum(1 for key in keys if old.get(key) != new.get(key) and (old.get(key) is not None or new.get(key) is not None))


def _metadata_summary(count):
    if not count: return ""
    noun = "cell" if count == 1 else "cells"
    return f"Ignored nbskill metadata changes in {count} {noun}."


@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str|None = "HEAD",  # First git ref; use None for working tree
    ref_b: str|None = None,  # Second git ref; defaults to working tree
    adds: bool = True,  # Include code cells added in ref_b
    changes: bool = True,  # Include changed code cells
    dels: bool = False,  # Include deleted code cells
):
    "Print nbdev-style diffs for code cells only; summarize nbskill metadata-only changes."
    ref_a, ref_b = none_if_string(ref_a), none_if_string(ref_b)
    if msg := (_git_ref_path_error(path, ref_a) or _git_ref_path_error(path, ref_b)):
        cli_error(msg)
    try:
        old, new = nbs_pair(path, ref_a=ref_a, ref_b=ref_b, f=code_source)
    except Exception as exc:
        detail = str(exc)
        hint = (
            f"Could not diff {path!r} against {ref_a!r}. "
            "The notebook may be new relative to that git ref, or the repository may not have a HEAD commit yet. "
            "Commit the notebook first, or pass --ref_a None to compare against the working tree."
        )
        if detail: hint += f"\nUnderlying error: {detail}"
        cli_error(hint)
    old = {cid: src for cid, src in old.items() if src is not None}
    new = {cid: src for cid, src in new.items() if src is not None}
    blocks = []
    if adds:    blocks += [(cid, source_diff("", new[cid])) for cid in new if cid not in old]
    if changes: blocks += [(cid, source_diff(old[cid], new[cid])) for cid in new if cid in old and new[cid] != old[cid]]
    if dels:    blocks += [(cid, source_diff(old[cid], "")) for cid in old if cid not in new]
    text = "\n\n".join(f"--- code cell {cid} ---\n{diff}" for cid, diff in blocks if diff.strip())
    metadata_summary = _metadata_summary(_nbskill_metadata_change_count(path, ref_a, ref_b))
    if text and metadata_summary: report = f"{text}\n\n{metadata_summary}"
    elif text: report = text
    elif metadata_summary: report = f"No code cell changes\n{metadata_summary}"
    else: report = "No code cell changes"
    print(report)
    return cli_return(report)

In [ ]:
path = demo_path("04_review_no_git.ipynb")
try:
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    try:
        diff_nb(str(path))
    except (SystemExit, ValueError) as exc:
        if isinstance(exc, SystemExit): assert exc.code == 1
        else:
            msg = str(exc)
            assert "No git repository" in msg or "Could not find notebook" in msg
finally:
    remove_demo_path(path)

root = demo_path("04_review_git")
try:
    root.mkdir()
    path = root / "demo.ipynb"
    _write_nb(new_nb([mk_cell("x = 1", cell_type="code")]), path)
    _subprocess.run(["git", "init"], cwd=root, check=True, capture_output=True)
    _subprocess.run(["git", "add", "demo.ipynb"], cwd=root, check=True, capture_output=True)
    _subprocess.run([
        "git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com",
        "commit", "-m", "base",
    ], cwd=root, check=True, capture_output=True)
    nb = _read_nb(path)
    nb.cells[0].metadata["nbskill"] = {"cell_type": "code", "semantic_types": [], "source_hash": "demo"}
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        diff_nb(str(path))
    text = out.getvalue()
    assert "No code cell changes" in text
    assert "Ignored nbskill metadata changes in 1 cell" in text
finally:
    remove_demo_path(root)

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import os as _os

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.foundation import remove_demo_path
from nbskill.review import (
    _format_global_usage_summary, _format_notebook_style_report, style_check,
)

path = demo_path("04_review_style.ipynb")
remove_demo_path(path)
try:
    long_source = "\n".join([f"x{i} = {i}" for i in range(21)])
    nb = new_nb([
        mk_cell("#| export\ndef a():\n    pass\ndef b():\n    pass\ndef c():\n    pass", cell_type="code"),
        mk_cell(long_source, cell_type="code"),
        mk_cell("assert 1 == 1\nassert 2 == 2", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("#| export\nimport os", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("import sys", cell_type="code"),
        mk_cell("result = later_helper()", cell_type="code"),
        mk_cell("def later_helper():\n    return 1", cell_type="code"),
        mk_cell("def loader():\n    return MissingPath('x')", cell_type="code"),
    ])
    _write_nb(nb, path)
    report = _format_notebook_style_report(path)
    assert "large-cell" in report
    assert "multi-problem-test" in report
    assert "unclean-cell" in report
    assert "scope=exported" in report
    assert "scope=internal" in report
    assert "cell-order" in report
    assert "missing-import" in report

    custom_map = demo_path("04_review_errors.json")
    old_map = _os.environ.get("NBSKILL_FAILURE_MAP")
    try:
        _os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
        with _redirect_stdout(_StringIO()):
            style_check(str(path), delete_after_output=True)
        assert "usage:" in _format_global_usage_summary()
        assert not custom_map.exists()
    finally:
        if old_map is None:
            _os.environ.pop("NBSKILL_FAILURE_MAP", None)
        else:
            _os.environ["NBSKILL_FAILURE_MAP"] = old_map

    try:
        with _redirect_stdout(_StringIO()):
            style_check(str(path), strict=True)
    except SystemExit as exc:
        assert exc.code
    else:
        raise AssertionError("strict style_check should exit for notebook hygiene problems")
finally:
    remove_demo_path(path)


In [ ]:
assert code_source(_mk_cell("plain docs", cell_type="markdown")) is None
assert code_source(_mk_cell("answer = 42", cell_type="code")) == "answer = 42"